In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy qiskit-addon-cutting
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Inizia con il circuit cutting tramite wire cut


<details>
<summary><b>Versioni dei pacchetti</b></summary>

Il codice in questa pagina è stato sviluppato con i seguenti requisiti.
Si consiglia di usare queste versioni o versioni più recenti.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
qiskit-aer~=0.17
qiskit-addon-cutting~=0.10.0
```
</details>
Questa guida mostra un esempio pratico di wire cut con il pacchetto `qiskit-addon-cutting`. Illustra come ricostruire i valori di aspettativa di un circuito a sette qubit tramite wire cutting.

Un wire cut è rappresentato in questo pacchetto come un'istruzione a due qubit [`Move`](https://docs.quantum.ibm.com/api/qiskit-addon-cutting/instructions-move), definita come un reset del secondo qubit su cui agisce l'istruzione, seguito da uno swap di entrambi i qubit. Questa operazione equivale a trasferire lo stato del primo qubit nel secondo qubit, scartando simultaneamente lo stato in ingresso del secondo qubit.

Il pacchetto è progettato per essere coerente con il modo in cui devi trattare i wire cut quando operi su qubit fisici. Ad esempio, un wire cut può prendere lo stato del qubit fisico $n$ e continuarlo come qubit fisico $m$ dopo il taglio. Puoi pensare all'"instruction cutting" come a un framework unificato per considerare sia i wire cut sia i gate cut all'interno dello stesso formalismo (poiché un wire cut è semplicemente un'istruzione [`Move`](https://docs.quantum.ibm.com/api/qiskit-addon-cutting/instructions-move) tagliata). Usare questo framework per il wire cutting consente inoltre il riutilizzo dei qubit, illustrato nella sezione su [come tagliare i wire manualmente](#cut-wires-using-the-low-level-move-instruction).

L'istruzione a singolo qubit [`CutWire`](https://docs.quantum.ibm.com/api/qiskit-addon-cutting/instructions-cut-wire) offre un'interfaccia più astratta e semplice per lavorare con i wire cut. Ti permette di indicare ad alto livello dove nel circuito deve essere effettuato il taglio, lasciando che l'addon per il circuit cutting inserisca le opportune istruzioni [`Move`](https://docs.quantum.ibm.com/api/qiskit-addon-cutting/instructions-move) al posto tuo.

L'esempio seguente mostra la ricostruzione del valore di aspettativa dopo un wire cutting. Creerai un circuito con diverse porte non locali e definirai le osservabili da stimare.

In [1]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.quantum_info import SparsePauliOp
from qiskit_ibm_runtime.fake_provider import FakeManilaV2
from qiskit_ibm_runtime import SamplerV2, Batch
from qiskit_aer.primitives import EstimatorV2

from qiskit_addon_cutting.instructions import Move, CutWire
from qiskit_addon_cutting import (
    partition_problem,
    generate_cutting_experiments,
    cut_wires,
    expand_observables,
    reconstruct_expectation_values,
)


qc_0 = QuantumCircuit(7)
for i in range(7):
    qc_0.rx(np.pi / 4, i)
qc_0.cx(0, 3)
qc_0.cx(1, 3)
qc_0.cx(2, 3)
qc_0.cx(3, 4)
qc_0.cx(3, 5)
qc_0.cx(3, 6)
qc_0.cx(0, 3)
qc_0.cx(1, 3)
qc_0.cx(2, 3)

# Define observable
observable = SparsePauliOp(["ZIIIIII", "IIIZIII", "IIIIIIZ"])

# Draw circuit
qc_0.draw("mpl")

<Image src="../docs/images/guides/qiskit-addons-cutting-wires/extracted-outputs/b481ef2d-3912-4eac-9755-335e8f5db886-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/qiskit-addons-cutting-wires/extracted-outputs/b481ef2d-3912-4eac-9755-335e8f5db886-0.svg)

## Taglia i wire con l'istruzione ad alto livello `CutWire`
Successivamente, effettua i wire cut usando l'istruzione a singolo qubit [`CutWire`](https://docs.quantum.ibm.com/api/qiskit-addon-cutting/instructions-cut-wire) sul qubit $q_3$. Una volta che i sottoesperimenti sono pronti per l'esecuzione, usa la funzione [`cut_wires()`](https://docs.quantum.ibm.com/api/qiskit-addon-cutting/qiskit-addon-cutting#cut_wires) per trasformare le istruzioni `CutWire` in istruzioni [`Move`](https://docs.quantum.ibm.com/api/qiskit-addon-cutting/instructions-move) su qubit appena allocati.

In [2]:
qc_1 = QuantumCircuit(7)
for i in range(7):
    qc_1.rx(np.pi / 4, i)
qc_1.cx(0, 3)
qc_1.cx(1, 3)
qc_1.cx(2, 3)
qc_1.append(CutWire(), [3])
qc_1.cx(3, 4)
qc_1.cx(3, 5)
qc_1.cx(3, 6)
qc_1.append(CutWire(), [3])
qc_1.cx(0, 3)
qc_1.cx(1, 3)
qc_1.cx(2, 3)

qc_1.draw("mpl")

<Image src="../docs/images/guides/qiskit-addons-cutting-wires/extracted-outputs/9bac1915-316b-49d0-a1a1-145047679530-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/qiskit-addons-cutting-wires/extracted-outputs/9bac1915-316b-49d0-a1a1-145047679530-0.svg)

> **Info:** Quando un circuito viene espanso tramite uno o più wire cut, l'osservabile deve essere aggiornata per tenere conto dei qubit aggiuntivi introdotti. Il pacchetto `qiskit-addon-cutting` mette a disposizione la funzione di utilità [`expand_observables()`](https://docs.quantum.ibm.com/api/qiskit-addon-cutting/qiskit-addon-cutting#expand_observables), che accetta oggetti `PauliList` e i circuiti originale ed espanso come argomenti, e restituisce una nuova `PauliList`.
> 
>     La `PauliList` restituita non conterrà alcuna informazione sui coefficienti dell'osservabile originale, ma questi possono essere ignorati fino alla ricostruzione del valore di aspettativa finale.

In [3]:
# Transform CutWire instructions to Move instructions
qc_2 = cut_wires(qc_1)

# Expand the observable to match the new circuit size
expanded_observable = expand_observables(observable.paulis, qc_0, qc_2)
print(f"Expanded Observable: {expanded_observable}")
qc_2.draw("mpl")

Expanded Observable: ['ZIIIIIIII', 'IIIZIIIII', 'IIIIIIIIZ']


<Image src="../docs/images/guides/qiskit-addons-cutting-wires/extracted-outputs/d398b397-0167-4db9-97ae-6ea502dc43e3-1.svg" alt="Output of the previous code cell" />

### Partition the circuit and observable

Now the problem can be separated into partitions. This is accomplished using the [`partition_problem()`](/docs/api/qiskit-addon-cutting/qiskit-addon-cutting#partition_problem) function with an optional set of partition labels to specify how to separate the circuit. Qubits sharing a common partition label are grouped together, and any non-local gates spanning more than one partition are cut.

If no partition labels are provided, then the partitioning will be automatically determined based on the connectivity of the circuit. Read the next section on [cutting wires manually](#cut-wires-using-the-low-level-move-instruction) for more information on including partition labels.

In [4]:
partitioned_problem = partition_problem(
    circuit=qc_2,
    observables=expanded_observable,
)
subcircuits = partitioned_problem.subcircuits
subobservables = partitioned_problem.subobservables
bases = partitioned_problem.bases

print(f"Subobservables to measure: \n{subobservables}\n")
print(f"Sampling overhead: {np.prod([basis.overhead for basis in bases])}")
subcircuits[0].draw("mpl")

Subobservables to measure: 
{0: PauliList(['IIIII', 'ZIIII', 'IIIIZ']), 1: PauliList(['ZIII', 'IIII', 'IIII'])}

Sampling overhead: 256.0


<Image src="../docs/images/guides/qiskit-addons-cutting-wires/extracted-outputs/5fb034f2-da8a-4f4d-ab9b-c990593e04fc-1.svg" alt="Output of the previous code cell" />

In [5]:
subcircuits[1].draw("mpl")

<Image src="../docs/images/guides/qiskit-addons-cutting-wires/extracted-outputs/d0e86f81-7c7e-4ccf-951c-9cd039135dc9-0.svg" alt="Output of the previous code cell" />

In this partitioning scheme, you have cut two wires, resulting in a sampling overhead of $4^4$.

### Generate subexperiments to execute and post-process results

To estimate the expectation value of the full-sized circuit, several subexperiments are generated from the decomposed gates' joint quasi-probability distribution and then executed on one (or more) QPUs. The [`generate_cutting_experiments`](/docs/api/qiskit-addon-cutting/qiskit-addon-cutting#generate_cutting_experiments) method does this by ingesting arguments for the `subcircuits` and `subobservables` dictionaries you created above, as well as for the number of samples to take from the distribution.

<Admonition type="note" title="Note about the number of samples">
    The `num_samples` argument specifies how many samples to draw from the quasi-probability distribution and determines the accuracy of the coefficients used for the reconstruction. Passing infinity (`np.inf`) ensures all coefficients are calculated exactly. Read the API docs on [generating weights](/docs/api/qiskit-addon-cutting/qpd#generate_qpd_weights) and [generating cutting experiments](/docs/api/qiskit-addon-cutting/qiskit-addon-cutting#generate_cutting_experiments) for more information.
</Admonition>

In [6]:
# Generate subexperiments
subexperiments, coefficients = generate_cutting_experiments(
    circuits=subcircuits, observables=subobservables, num_samples=np.inf
)

# Set a backend to use and transpile the subexperiments
backend = FakeManilaV2()
pass_manager = generate_preset_pass_manager(
    optimization_level=1, backend=backend
)
isa_subexperiments = {
    label: pass_manager.run(partition_subexpts)
    for label, partition_subexpts in subexperiments.items()
}

# Submit each partition's subexperiments to the Qiskit Runtime Sampler
# primitive, in a single batch so that the jobs will run back-to-back.
with Batch(backend=backend) as batch:
    sampler = SamplerV2(mode=batch)
    jobs = {
        label: sampler.run(subsystem_subexpts, shots=2**12)
        for label, subsystem_subexpts in isa_subexperiments.items()
    }
    # Retrieve results
    results = {label: job.result() for label, job in jobs.items()}

Lastly, the expectation value of the full circuit can be reconstructed using the [`reconstruct_expectation_values()`](/docs/api/qiskit-addon-cutting/qiskit-addon-cutting#reconstruct_expectation_values) method.


The code block below reconstructs the results and compares them with the exact expectation value.

In [7]:
reconstructed_expval_terms = reconstruct_expectation_values(
    results,
    coefficients,
    subobservables,
)
# Apply the coefficients of the original observable
reconstructed_expval = np.dot(reconstructed_expval_terms, observable.coeffs)


# Compute the exact expectation value using the `qiskit_aer` package.
estimator = EstimatorV2()
exact_expval = estimator.run([(qc_0, observable)]).result()[0].data.evs
print(
    f"Reconstructed expectation value: {np.real(np.round(reconstructed_expval, 8))}"
)
print(f"Exact expectation value: {np.round(exact_expval, 8)}")
print(
    f"Error in estimation: {np.real(np.round(reconstructed_expval-exact_expval, 8))}"
)
print(
    f"Relative error in estimation: {np.real(np.round((reconstructed_expval-exact_expval) / exact_expval, 8))}"
)

Reconstructed expectation value: 1.45965266
Exact expectation value: 1.59099026
Error in estimation: -0.1313376
Relative error in estimation: -0.08255085


![Output of the previous code cell](../docs/images/guides/qiskit-addons-cutting-wires/extracted-outputs/d0e86f81-7c7e-4ccf-951c-9cd039135dc9-0.svg)

Con questo schema di partizionamento, hai tagliato due wire, con un overhead di campionamento di $4^4$.

### Genera i sottoesperimenti da eseguire e post-elabora i risultati

Per stimare il valore di aspettativa del circuito completo, vengono generati diversi sottoesperimenti dalla distribuzione di quasi-probabilità congiunta delle porte decomposte, che vengono poi eseguiti su uno o più QPU. Il metodo [`generate_cutting_experiments`](https://docs.quantum.ibm.com/api/qiskit-addon-cutting/qiskit-addon-cutting#generate_cutting_experiments) esegue questa operazione accettando come argomenti i dizionari `subcircuits` e `subobservables` creati in precedenza, nonché il numero di campioni da estrarre dalla distribuzione.

> **Note:** L'argomento `num_samples` specifica quanti campioni estrarre dalla distribuzione di quasi-probabilità e determina la precisione dei coefficienti usati per la ricostruzione. Passare infinito (`np.inf`) garantisce che tutti i coefficienti vengano calcolati in modo esatto. Leggi la documentazione API su come [generare i pesi](https://docs.quantum.ibm.com/api/qiskit-addon-cutting/qpd#generate_qpd_weights) e su come [generare gli esperimenti di cutting](https://docs.quantum.ibm.com/api/qiskit-addon-cutting/qiskit-addon-cutting#generate_cutting_experiments) per ulteriori informazioni.

In [8]:
qc_1 = QuantumCircuit(8)
for i in [*range(4), *range(5, 8)]:
    qc_1.rx(np.pi / 4, i)
qc_1.cx(0, 3)
qc_1.cx(1, 3)
qc_1.cx(2, 3)
qc_1.append(Move(), [3, 4])
qc_1.cx(4, 5)
qc_1.cx(4, 6)
qc_1.cx(4, 7)
qc_1.append(Move(), [4, 3])
qc_1.cx(0, 3)
qc_1.cx(1, 3)
qc_1.cx(2, 3)

# Expand observable
observable_expanded = SparsePauliOp(["ZIIIIIII", "IIIIZIII", "IIIIIIIZ"])
qc_1.draw("mpl")

<Image src="../docs/images/guides/qiskit-addons-cutting-wires/extracted-outputs/15461a2c-85a9-4cb2-a632-b9597ccbc4bd-0.svg" alt="Output of the previous code cell" />

Infine, il valore di aspettativa del circuito completo può essere ricostruito usando il metodo [`reconstruct_expectation_values()`](https://docs.quantum.ibm.com/api/qiskit-addon-cutting/qiskit-addon-cutting#reconstruct_expectation_values).

Il blocco di codice seguente ricostruisce i risultati e li confronta con il valore di aspettativa esatto.

In [9]:
partitioned_problem = partition_problem(
    circuit=qc_1,
    partition_labels="AAAABBBB",
    observables=observable_expanded.paulis,
)
subcircuits = partitioned_problem.subcircuits
subobservables = partitioned_problem.subobservables
bases = partitioned_problem.bases

print(f"Subobservables to measure: \n{subobservables}\n")
print(f"Sampling overhead: {np.prod([basis.overhead for basis in bases])}")
subcircuits["A"].draw("mpl")

Subobservables to measure: 
{'A': PauliList(['IIII', 'ZIII', 'IIIZ']), 'B': PauliList(['ZIII', 'IIII', 'IIII'])}

Sampling overhead: 256.0


<Image src="../docs/images/guides/qiskit-addons-cutting-wires/extracted-outputs/2139745a-bdc3-40bd-bd6f-d26d2a5b5b14-1.svg" alt="Output of the previous code cell" />

In [10]:
subcircuits["B"].draw("mpl")

<Image src="../docs/images/guides/qiskit-addons-cutting-wires/extracted-outputs/4aeb3f1f-a55e-49c4-a7bd-837132429ee1-0.svg" alt="Output of the previous code cell" />

> **Caution:** Per ricostruire con precisione il valore di aspettativa, i coefficienti dell'osservabile originale (che sono diversi dall'output di `generate_cutting_experiments()`) devono essere applicati all'output della ricostruzione, poiché questa informazione viene persa quando vengono generati gli esperimenti di cutting o quando l'osservabile viene espansa.
> 
>     In genere questi coefficienti possono essere applicati tramite `numpy.dot()`, come mostrato in precedenza.
## Taglia i wire con l'istruzione a basso livello `Move`
Un limite dell'uso dell'istruzione di livello superiore `CutWire` è che non consente il riutilizzo dei qubit. Se lo desideri per un esperimento di cutting, puoi invece posizionare manualmente le istruzioni [`Move`](https://docs.quantum.ibm.com/api/qiskit-addon-cutting/instructions-move). Tuttavia, poiché l'istruzione `Move` scarta lo stato del qubit di destinazione, è importante che questo qubit non condivida alcun entanglement con il resto del sistema. In caso contrario, l'operazione di reset causerà un collasso parziale dello stato del circuito dopo il wire cut.

Il blocco di codice seguente esegue un wire cut sul qubit $q_3$ per lo stesso circuito di esempio mostrato in precedenza. La differenza qui è che puoi riutilizzare un qubit invertendo l'operazione `Move` nel punto in cui è stato effettuato il secondo wire cut (tuttavia, ciò non è sempre possibile e dipende dal circuito che si sta tagliando).